# RAG Pipeline — Echolabs
**Assignment 8: Rag Pipeline, System Experiments, and Failure Analysis**

In [1]:
import re
import time
import numpy as np
import pandas as pd
import torch
from typing import List, Dict
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

## Step 1.1: Define Your Knowledge Base

**Dataset:** [`knkarthick/dialogsum`](https://huggingface.co/datasets/knkarthick/dialogsum)

In [2]:
df = load_dataset("knkarthick/dialogsum", split="train")
sampled = df.select(range(40))

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [3]:
# Build the full paragraph
PARAGRAPHS = []
for i, row in enumerate(sampled):
    text = row['dialogue'].strip()
    word_count = len(text.split())
    if word_count >= 80: 
        PARAGRAPHS.append({
            'text':       text,
            'word_count': word_count,
            'speaker':    'Multiple Speakers',
            'title':      f"Dialogue {i+1}",
            'para_idx':   i,
            'summary':    row['summary']
        })

In [4]:
# Preview one paragraph
sample = PARAGRAPHS[2]
print(sample['text'])

#Person1#: Excuse me, did you see a set of keys?
#Person2#: What kind of keys?
#Person1#: Five keys and a small foot ornament.
#Person2#: What a shame! I didn't see them.
#Person1#: Well, can you help me look for it? That's my first time here.
#Person2#: Sure. It's my pleasure. I'd like to help you look for the missing keys.
#Person1#: It's very kind of you.
#Person2#: It's not a big deal.Hey, I found them.
#Person1#: Oh, thank God! I don't know how to thank you, guys.
#Person2#: You're welcome.


## Step 1.2: Text Representation & Chunking

### How Paragraphs Are Defined
Paragraphs are sentence boundary safe thought segments extracted from each talk's transcript. Stage directions are stripped first. Sentences are accumulated until the 200-word ceiling is reached; anything under 80 words is merged forward. No sentence is ever split mid-way across a boundary.

### How Overlap Is Implemented (Strategy 2)
A sliding window of size 2 advances by 1 paragraph at a time. Chunk *i* = paragraphs [*i*, *i+1*]; chunk *i+1* = paragraphs [*i+1*, *i+2*]. Paragraph *i+1* appears in both consecutive chunks — this is the overlap. With `window=2, step=1` the overlap rate is 50%. This ensures no content near a chunk boundary is lost from retrieval.

| Strategy | Unit | Description |
|---|---|---|
| **Fixed-Length** | Words | Pack sentences to a ~150-word target; sentence boundaries always respected |
| **Overlapping Paragraph** | Paragraphs | 2-paragraph sliding window, step=1; adjacent chunks share 1 paragraph (50% overlap) |
| **Hybrid/Strategic** | Talk sections | Group 3 consecutive paragraphs from the same talk into one section chunk |

In [5]:
# ----------- Strategy 1: Fixed-Length Chunking -----------

def fixed_length_chunking(paragraphs: List[Dict], target_words: int = 150) -> List[Dict]:
    full_text = ' '.join(p['text'] for p in paragraphs)
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z"])', full_text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks, current_sents, current_wc = [], [], 0

    for sent in sentences:
        sw = len(sent.split())
        if current_wc + sw > target_words and current_sents:
            chunks.append({
                'text':       ' '.join(current_sents),
                'word_count': current_wc,
                'strategy':   'fixed_length',
                'chunk_id':   len(chunks)
            })
            current_sents, current_wc = [], 0
        current_sents.append(sent)
        current_wc += sw

    if current_sents:
        chunks.append({
            'text':       ' '.join(current_sents),
            'word_count': current_wc,
            'strategy':   'fixed_length',
            'chunk_id':   len(chunks)
        })
    return chunks


fixed_chunks = fixed_length_chunking(PARAGRAPHS, target_words=150)
print(f"Fixed-Length  →  {len(fixed_chunks)} chunks")
print(f"Avg words     :  {np.mean([c['word_count'] for c in fixed_chunks]):.1f}")
print()
print("Sample chunk [0]:")
print("-" * 60)
print(fixed_chunks[0]['text'])

Fixed-Length  →  33 chunks
Avg words     :  133.9

Sample chunk [0]:
------------------------------------------------------------
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know.


In [6]:
# ── Strategy 2: Overlapping Paragraph Chunking ────────────────────────────────

def overlapping_paragraph_chunking(paragraphs: List[Dict],
                                    window: int = 2,
                                    step:   int = 1) -> List[Dict]:
    chunks = []
    for start in range(0, len(paragraphs) - window + 1, step):
        window_paras = paragraphs[start : start + window]
        combined_text = '\n\n'.join(p['text'] for p in window_paras)
        chunks.append({
            'text':            combined_text,
            'word_count':      len(combined_text.split()),
            'strategy':        'overlapping_paragraph',
            'chunk_id':        len(chunks),
            'source_para_ids': list(range(start, start + window)),
            'speaker':         window_paras[0]['speaker'],
            'title':           window_paras[0]['title'],
        })
    return chunks


overlap_chunks = overlapping_paragraph_chunking(PARAGRAPHS, window=2, step=1)
print(f"Overlapping Paragraph  →  {len(overlap_chunks)} chunks")
print(f"Avg words              :  {np.mean([c['word_count'] for c in overlap_chunks]):.1f}")
print(f"Overlap                :  1 paragraph shared per adjacent pair (50%)")
print()
print(f"Chunk 0 → paras {overlap_chunks[0]['source_para_ids']}")
print(f"Chunk 1 → paras {overlap_chunks[1]['source_para_ids']}")
print(f"  → Para 1 appears in BOTH (the overlap)")

Overlapping Paragraph  →  31 chunks
Avg words              :  274.4
Overlap                :  1 paragraph shared per adjacent pair (50%)

Chunk 0 → paras [0, 1]
Chunk 1 → paras [1, 2]
  → Para 1 appears in BOTH (the overlap)


In [7]:
# ── Strategy 3: Hybrid / Strategic Chunking ───────────────────────────────────

def hybrid_strategic_chunking(paragraphs: List[Dict],
                               paras_per_section: int = 3) -> List[Dict]:
    chunks = []
    total_sections = (len(paragraphs) + paras_per_section - 1) // paras_per_section

    for i in range(0, len(paragraphs), paras_per_section):
        section = paragraphs[i : i + paras_per_section]
        combined = '\n\n'.join(p['text'] for p in section)
        section_num = i // paras_per_section + 1
        chunks.append({
            'text':       combined,
            'word_count': len(combined.split()),
            'strategy':   'hybrid_strategic',
            'chunk_id':   len(chunks),
            'title':      f"Dialogues {i+1}–{i+len(section)}",
            'section':    f"Section {section_num}/{total_sections}",
        })
    return chunks

hybrid_chunks = hybrid_strategic_chunking(PARAGRAPHS, paras_per_section=3)
print(f"Hybrid/Strategic  →  {len(hybrid_chunks)} chunks")
print(f"Avg words         :  {np.mean([c['word_count'] for c in hybrid_chunks]):.1f}")
print()
print("Chunks by section:")
for c in hybrid_chunks:
    print(f"  [{c['chunk_id']:2d}]  {c['title']:<25}  {c['section']}  ({c['word_count']} words)")

Hybrid/Strategic  →  11 chunks
Avg words         :  401.8

Chunks by section:
  [ 0]  Dialogues 1–3              Section 1/11  (396 words)
  [ 1]  Dialogues 4–6              Section 2/11  (425 words)
  [ 2]  Dialogues 7–9              Section 3/11  (328 words)
  [ 3]  Dialogues 10–12            Section 4/11  (469 words)
  [ 4]  Dialogues 13–15            Section 5/11  (303 words)
  [ 5]  Dialogues 16–18            Section 6/11  (386 words)
  [ 6]  Dialogues 19–21            Section 7/11  (412 words)
  [ 7]  Dialogues 22–24            Section 8/11  (369 words)
  [ 8]  Dialogues 25–27            Section 9/11  (516 words)
  [ 9]  Dialogues 28–30            Section 10/11  (559 words)
  [10]  Dialogues 31–32            Section 11/11  (257 words)


## Step 1.3: Embedding Pipeline
| Size | Model | Dimensions | License |
|---|---|---|---|
| **Small** | [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | 384 | Apache 2.0 |
| **Medium** | [`sentence-transformers/all-mpnet-base-v2`](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | 768 | Apache 2.0 |
| **Large** | [`BAAI/bge-large-en-v1.5`](https://huggingface.co/BAAI/bge-large-en-v1.5) | 1024 | MIT |

In [8]:
EMBEDDING_MODELS = {
    'small':  {'hf_name': 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1',  'dims': 384},
    'medium': {'hf_name': 'sentence-transformers/multi-qa-mpnet-base-cos-v1',  'dims': 768},
    'large':  {'hf_name': 'BAAI/bge-large-en-v1.5',                   'dims': 1024},
}

loaded_embed_models = {}
for size, info in EMBEDDING_MODELS.items():
    loaded_embed_models[size] = SentenceTransformer(info['hf_name'])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [9]:
def embed_chunks(chunks: List[Dict], model: SentenceTransformer,
                 model_size: str, batch_size: int = 16) -> Dict:
    texts = [c['text'] for c in chunks]
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return {
        'embeddings': np.array(embeddings, dtype=np.float32),
        'chunks':     chunks,
        'model_size': model_size
    }


# Build index_store[strategy][model_size]
all_chunks = {
    'fixed_length':          fixed_chunks,
    'overlapping_paragraph': overlap_chunks,
    'hybrid_strategic':      hybrid_chunks,
}

index_store = {}
for strategy_name, chunks in all_chunks.items():
    index_store[strategy_name] = {}
    for size, model in loaded_embed_models.items():
        result = embed_chunks(chunks, model, size)
        index_store[strategy_name][size] = result
        print(f"  ✓ {strategy_name:<28} | {size:<6} | shape={result['embeddings'].shape}")

  ✓ fixed_length                 | small  | shape=(33, 384)
  ✓ fixed_length                 | medium | shape=(33, 768)
  ✓ fixed_length                 | large  | shape=(33, 1024)
  ✓ overlapping_paragraph        | small  | shape=(31, 384)
  ✓ overlapping_paragraph        | medium | shape=(31, 768)
  ✓ overlapping_paragraph        | large  | shape=(31, 1024)
  ✓ hybrid_strategic             | small  | shape=(11, 384)
  ✓ hybrid_strategic             | medium | shape=(11, 768)
  ✓ hybrid_strategic             | large  | shape=(11, 1024)


## Step 1.4: Retrieval System

**Flow:**
1. Encode the query with the selected embedding model
2. Compute cosine similarity between the query vector and all stored chunk vectors
3. Return the top-*k* chunks by descending similarity score

In [10]:
def retrieve(
    query:      str,
    strategy:   str = 'hybrid_strategic',
    model_size: str = 'medium',
    top_k:      int = 3
) -> List[Dict]:
    embed_model = loaded_embed_models[model_size]
    query_vec = embed_model.encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    )

    store = index_store[strategy][model_size]
    chunk_embeddings = store['embeddings']
    chunks = store['chunks']

    scores = cosine_similarity(query_vec, chunk_embeddings)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        entry = dict(chunks[idx])
        entry['score'] = float(scores[idx])
        results.append(entry)
    return results


# ── Smoke test ────────────────────────────────────────────────────────────────
test_q = "What are people discussing about work and scheduling plans?"
results = retrieve(test_q, strategy='hybrid_strategic', model_size='medium', top_k=3)

print(f"Query: {test_q}")
print("=" * 70)
for i, r in enumerate(results):
    print(f"\nRank {i+1}  |  score={r['score']:.4f}  |  {r.get('title', r['chunk_id'])}")
    print(r['text'][:250] + '...')

Query: What are people discussing about work and scheduling plans?

Rank 1  |  score=0.2663  |  Dialogues 28–30
#Person1#: I'm tired of watching television. Let's go to cinema to- night.
#Person2#: All right. Do you want to go downtown? Or is there a good movie in the neighborhood?
#Person1#: I'd rather not spend a lot of money. What does the pa- per say about...

Rank 2  |  score=0.2272  |  Dialogues 22–24
#Person1#: Mr. White, I would like to give you notice that I will be leaving the company. It will be effective at the beginning of the next month.
#Person2#: Jessica, I am very sorry to hear that. Why are you leaving?
#Person1#: I've been offered ano...

Rank 3  |  score=0.2240  |  Dialogues 7–9
#Person1#: This is a good basic computer package. It's got a good CPU, 256 megabytes of RAM, and a DVD player.
#Person2#: Does it come with a modem?
#Person1#: Yes, it has a built-in modem. You just plug a phone line into the back of the computer.
#P...


## Step 1.5: Generation Pipeline

**Model:** `google/flan-t5-base`

In [11]:
GEN_MODEL_NAME = 'google/flan-t5-base'

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model     = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME).to(device)
gen_model.eval()

param_count = sum(p.numel() for p in gen_model.parameters())

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [13]:
def build_prompt(query: str, retrieved_chunks: List[Dict]) -> str:
    """Assemble the RAG prompt: instruction + labelled context blocks + question."""
    context_block = ''
    for i, chunk in enumerate(retrieved_chunks):
        section = chunk.get('section', f"Chunk {chunk['chunk_id']}")
        # Truncate to keep within flan-t5-base's 512-token input limit
        snippet = ' '.join(chunk['text'].split()[:250])
        context_block += f"[Context {i+1} — {section}]\n{snippet}\n\n"

    return (
        "Answer the following question using only the provided context. "
        "Be specific and concise.\n\n"
        + context_block.strip()
        + f"\n\nQuestion: {query}\nAnswer:"
    )


def generate_answer(prompt: str, max_new_tokens: int = 120) -> str:
    """Run flan-t5-base inference using beam search (matches A7 setup)."""
    inputs = gen_tokenizer(
        prompt, return_tensors='pt', max_length=512, truncation=True
    ).to(device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, num_beams=4, early_stopping=True
        )
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def rag_generate(
    query:      str,
    strategy:   str  = 'hybrid_strategic',
    model_size: str  = 'medium',
    top_k:      int  = 3,
    verbose:    bool = True
) -> Dict:
    """
    Full RAG pipeline: Query → Retrieve → Prompt → Generate.
    Returns a dict with query, strategy, model_size, retrieved_chunks,
    answer, and latency_sec — ready for the Part 2 evaluation table.
    """
    start     = time.time()
    retrieved = retrieve(query, strategy=strategy, model_size=model_size, top_k=top_k)
    prompt    = build_prompt(query, retrieved)
    answer    = generate_answer(prompt)
    latency   = round(time.time() - start, 2)

    if verbose:
        dims = EMBEDDING_MODELS[model_size]['dims']
        print(f"Query    : {query}")
        print(f"Strategy : {strategy}  |  Model: {model_size} ({dims}d)  |  Top-k: {top_k}")
        print(f"Latency  : {latency}s")
        print()
        print("Retrieved chunks:")
        for r in retrieved:
            print(f"  score={r['score']:.4f}  | {r.get('section', f'chunk_{r[chr(99)+chr(104)+chr(117)+chr(110)+chr(107)+chr(95)+chr(105)+chr(100)]}')  }")
            print(f"  Preview: {r['text'][:180]}...")
            print()
        print(f"Answer   : {answer}")

    return {
        'query':            query,
        'strategy':         strategy,
        'model_size':       model_size,
        'retrieved_chunks': retrieved,
        'answer':           answer,
        'latency_sec':      latency
    }

# Part 2: System Experiments
**3 embedding models × 3 chunking strategies = 9 configurations**

All experiments use `google/flan-t5-base` as the fixed generation model.  
Retrieval: cosine similarity, top-k = 3.

## Step 2.1 — Defining Test Queries

In [14]:
# 5 realistic queries related to the dialogsum knowledge base 
TEST_QUERIES = [
    "What are two people discussing about a medical check-up or health concern?",
    "How do characters in these dialogues handle a job resignation or workplace issue?",
    "What advice is given about buying or using a computer or electronic device?",
    "What do people plan to do together for entertainment or leisure this evening?",
    "How does someone ask for help finding a lost item or solving a problem?",
]

STRATEGIES  = ['fixed_length', 'overlapping_paragraph', 'hybrid_strategic']
MODEL_SIZES = ['small', 'medium', 'large']
TOP_K       = 3

print(f"Test queries  : {len(TEST_QUERIES)}")
print(f"Strategies    : {STRATEGIES}")
print(f"Embedding mdls: {MODEL_SIZES}")
print(f"Configurations: {len(STRATEGIES) * len(MODEL_SIZES)} (3 × 3)")
print(f"Total runs    : {len(TEST_QUERIES) * len(STRATEGIES) * len(MODEL_SIZES)}")

Test queries  : 5
Strategies    : ['fixed_length', 'overlapping_paragraph', 'hybrid_strategic']
Embedding mdls: ['small', 'medium', 'large']
Configurations: 9 (3 × 3)
Total runs    : 45


## Step 2.2 — Running All 9 Configurations & Build Evaluation Table

For every *(embedding model, chunking strategy, query)* combination we record:
- The retrieved chunks (text + similarity score)
- The generated answer
- Latency in seconds
- Manual quality scores (1–5) explained in the rubric below

**Scoring rubric**

| Score | Retrieved Context Quality | Answer Quality |
|-------|--------------------------|----------------|
| 5 | Highly relevant, no noise | Accurate, complete, well-phrased |
| 4 | Mostly relevant, minor noise | Mostly correct, minor gaps |
| 3 | Partially relevant (≥1 good chunk) | Partially correct |
| 2 | Mostly irrelevant | Vague or mostly wrong |
| 1 | Completely off-topic | Hallucinated / no answer |

In [15]:
import time, textwrap

raw_results = []   # stores full dicts with retrieved chunks

total = len(STRATEGIES) * len(MODEL_SIZES) * len(TEST_QUERIES)
run_n = 0

for strategy in STRATEGIES:
    for model_size in MODEL_SIZES:
        for query in TEST_QUERIES:
            run_n += 1
            dims = EMBEDDING_MODELS[model_size]['dims']
            print(f"[{run_n:02d}/{total}] {strategy} | {model_size} ({dims}d) | {query[:60]}...")

            result = rag_generate(
                query=query,
                strategy=strategy,
                model_size=model_size,
                top_k=TOP_K,
                verbose=False   # suppress per-run output; we display below
            )
            result['dims'] = dims
            raw_results.append(result)

print(f"\n✓ All {total} runs complete.")

[01/45] fixed_length | small (384d) | What are two people discussing about a medical check-up or h...
[02/45] fixed_length | small (384d) | How do characters in these dialogues handle a job resignatio...
[03/45] fixed_length | small (384d) | What advice is given about buying or using a computer or ele...
[04/45] fixed_length | small (384d) | What do people plan to do together for entertainment or leis...
[05/45] fixed_length | small (384d) | How does someone ask for help finding a lost item or solving...
[06/45] fixed_length | medium (768d) | What are two people discussing about a medical check-up or h...
[07/45] fixed_length | medium (768d) | How do characters in these dialogues handle a job resignatio...
[08/45] fixed_length | medium (768d) | What advice is given about buying or using a computer or ele...
[09/45] fixed_length | medium (768d) | What do people plan to do together for entertainment or leis...
[10/45] fixed_length | medium (768d) | How does someone ask for help finding a

In [16]:
# ── Display retrieved chunks for every run (required by assignment) ────────────

for res in raw_results:
    dims = EMBEDDING_MODELS[res['model_size']]['dims']
    print("=" * 80)
    print(f"STRATEGY : {res['strategy']}")
    print(f"EMBEDDING: {res['model_size']} ({dims}d)")
    print(f"QUERY    : {res['query']}")
    print(f"LATENCY  : {res['latency_sec']}s")
    print()
    print("  RETRIEVED CHUNKS:")
    for i, chunk in enumerate(res['retrieved_chunks']):
        title = chunk.get('title', chunk.get('section', f"chunk_{chunk['chunk_id']}"))
        print(f"    [{i+1}] score={chunk['score']:.4f} | {title}")
        wrapped = textwrap.fill(chunk['text'][:400], width=72, initial_indent='        ',
                                subsequent_indent='        ')
        print(wrapped + ("..." if len(chunk['text']) > 400 else ""))
        print()
    print(f"  ANSWER: {res['answer']}")
    print()

STRATEGY : fixed_length
EMBEDDING: small (384d)
QUERY    : What are two people discussing about a medical check-up or health concern?
LATENCY  : 11.04s

  RETRIEVED CHUNKS:
    [1] score=0.4739 | chunk_0
        #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here
        today? #Person2#: I found it would be a good idea to get a
        check-up. #Person1#: Yes, well, you haven't had one for 5 years.
        You should have one every year. #Person2#: I know. I figure as
        long as there is nothing wrong, why go see the doctor?
        #Person1#: Well, the best way to avoid serious illnesses is to
        find out about them early. So tr...

    [2] score=0.2798 | chunk_1
        You really should quit. #Person2#: I've tried hundreds of times,
        but I just can't seem to kick the habit. #Person1#: Well, we
        have classes and some medications that might help. I'll give you
        more information before you leave. #Person2#: Ok, thanks doctor.
        #Person1

In [17]:
# Layout: (strategy, model_size, query_index) → (context_quality, answer_quality)
# Scores are 1–5 per the rubric in Step 2.2

QUALITY_SCORES = {
    # --- fixed_length ---
    ('fixed_length', 'small',  0): (3, 2),   # chunk_0 correct, chunk_14 off-topic; answer repeats phrase
    ('fixed_length', 'small',  1): (3, 3),   # rank1 correct, ranks 2-3 tangential; answer captures resignation
    ('fixed_length', 'small',  2): (2, 1),   # only rank1 relevant; answer is "Doctor Hawkins" - wrong
    ('fixed_length', 'small',  3): (2, 1),   # no relevant chunk; answer "A glass of wine" - hallucination
    ('fixed_length', 'small',  4): (3, 2),   # rank1 mixed keys+vaccines; answer "I'd like to see that myself" - irrelevant
    ('fixed_length', 'medium', 0): (3, 2),   # rank1 correct, rank3 irrelevant restaurant; answer loops phrase
    ('fixed_length', 'medium', 1): (2, 1),   # correct chunk at rank3 not rank1; answer about girlfriend - wrong
    ('fixed_length', 'medium', 2): (2, 1),   # huge score gap, only rank1 useful; answer wrong
    ('fixed_length', 'medium', 3): (2, 1),   # chunks about past leisure not planning; answer "Movie" - lucky guess
    ('fixed_length', 'medium', 4): (3, 2),   # rank1 has keys buried with vaccines; answer "Yes, please" - vague
    ('fixed_length', 'large',  0): (4, 2),   # top 2 health-related, rank3 off-topic; answer repeats phrase
    ('fixed_length', 'large',  1): (4, 3),   # ranks 1-2 strongly on-topic; answer partially correct
    ('fixed_length', 'large',  2): (3, 2),   # rank1 correct, ranks 2-3 related but not computers; answer wrong
    ('fixed_length', 'large',  3): (3, 3),   # rank2 TV/baseball best match; answer "There's a baseball game on tonight"
    ('fixed_length', 'large',  4): (3, 3),   # rank1 has keys; answer references searching and getting help
    # --- overlapping_paragraph ---
    ('overlapping_paragraph', 'small',  0): (4, 2),  # top 2 health-related; answer just "vaccines"
    ('overlapping_paragraph', 'small',  1): (2, 1),  # rank1 correct, rank2 bus contamination; answer wrong
    ('overlapping_paragraph', 'small',  2): (2, 1),  # very low scores, all off-topic except rank1; answer "Judy Liao"
    ('overlapping_paragraph', 'small',  3): (3, 3),  # rank2 cinema dialogue correct; answer quotes cinema accurately
    ('overlapping_paragraph', 'small',  4): (3, 1),  # rank1 correct keys; answer about vaccines - model ignored best chunk
    ('overlapping_paragraph', 'medium', 0): (4, 2),  # top 2 correct, rank3 off-topic; answer "vaccines" - incomplete
    ('overlapping_paragraph', 'medium', 1): (2, 1),  # same bus contamination; answer wrong
    ('overlapping_paragraph', 'medium', 2): (2, 1),  # computer at rank2 not rank1, medium worse than small; answer wrong
    ('overlapping_paragraph', 'medium', 3): (3, 2),  # rank1 cinema correct; answer "So, how was your vacation?" - wrong
    ('overlapping_paragraph', 'medium', 4): (2, 1),  # correct chunk not retrieved; answer wrong
    ('overlapping_paragraph', 'large',  0): (4, 2),  # top 2 health-related, high confidence; answer just "vaccines"
    ('overlapping_paragraph', 'large',  1): (3, 1),  # rank1 correct, bus still contaminates; answer wrong
    ('overlapping_paragraph', 'large',  2): (3, 1),  # rank1 correct, ranks 2-3 off-topic; answer "Judy Liao"
    ('overlapping_paragraph', 'large',  3): (3, 2),  # rank1 cinema correct; answer "So, how was your vacation?" - ignores rank1
    ('overlapping_paragraph', 'large',  4): (3, 2),  # rank1 correct keys; answer not ideal but close
    # --- hybrid_strategic ---
    ('hybrid_strategic', 'small',  0): (2, 1),  # right section mixed with unrelated; answer "beautiful" - wrong
    ('hybrid_strategic', 'small',  1): (2, 1),  # relevant buried in section; answer wrong
    ('hybrid_strategic', 'small',  2): (1, 1),  # extremely low scores, computer buried; answer wrong
    ('hybrid_strategic', 'small',  3): (2, 1),  # cinema present but mixed; answer fragment - wrong
    ('hybrid_strategic', 'small',  4): (1, 1),  # keys dialogue entirely absent; answer wrong
    ('hybrid_strategic', 'medium', 0): (2, 1),  # answer "beautiful" - latches onto babies section
    ('hybrid_strategic', 'medium', 1): (2, 1),  # ranks 2-3 off-topic girlfriend; answer wrong
    ('hybrid_strategic', 'medium', 2): (1, 1),  # lowest scores of any config (0.07 rank3); answer total hallucination
    ('hybrid_strategic', 'medium', 3): (2, 1),  # cinema present but answer reads wrong part of section
    ('hybrid_strategic', 'medium', 4): (1, 1),  # no relevant chunk retrieved; answer wrong
    ('hybrid_strategic', 'large',  0): (2, 1),  # high scores but sections mix topics; answer "beautiful"
    ('hybrid_strategic', 'large',  1): (2, 1),  # rank1 correct section but answer wrong; score inflation
    ('hybrid_strategic', 'large',  2): (2, 1),  # computer present but diluted; answer hallucination
    ('hybrid_strategic', 'large',  3): (3, 1),  # best hybrid retrieval for Q4 - cinema+movies; answer still wrong
    ('hybrid_strategic', 'large',  4): (1, 1),  # keys dialogue absent, ranked below other sections; answer wrong
}

print(f"Quality score entries: {len(QUALITY_SCORES)}  (expected {total})")

Quality score entries: 45  (expected 45)


In [20]:
# ── Aggregated summary per configuration (avg across 5 queries) ───────────────

summary = (
    eval_df
    .groupby(['Embedding Model', 'Chunking Strategy'])
    .agg(
        Avg_Context_Quality  =('Context Quality (1-5)', 'mean'),
        Avg_Answer_Quality   =('Answer Quality (1-5)',  'mean'),
        Avg_Latency_s        =('Latency (s)',           'mean'),
        N                    =('Query',                 'count'),
    )
    .round(2)
    .reset_index()
    .sort_values(['Avg_Answer_Quality', 'Avg_Context_Quality'], ascending=False)
)
print("Summary — averages per configuration (over 5 queries):")
summary

Summary — averages per configuration (over 5 queries):


,Embedding Model,Chunking Strategy,Avg_Context_Quality,Avg_Answer_Quality,Avg_Latency_s,N
0,large (1024d),fixed_length,3.4,2.6,3.85,5
6,small (384d),fixed_length,2.6,1.8,4.58,5
2,large (1024d),overlapping_paragraph,3.2,1.6,1.25,5
8,small (384d),overlapping_paragraph,2.8,1.6,2.32,5
5,medium (768d),overlapping_paragraph,2.6,1.4,1.19,5
3,medium (768d),fixed_length,2.4,1.4,2.99,5
1,large (1024d),hybrid_strategic,2.0,1.0,1.23,5
4,medium (768d),hybrid_strategic,1.6,1.0,1.35,5
7,small (384d),hybrid_strategic,1.6,1.0,1.20,5


## Step 2.3 — Compare Embedding Models

How does embedding size affect retrieval quality and answer quality?

In [21]:
embed_compare = (
    eval_df
    .groupby('Embedding Model')
    .agg(
        Avg_Context_Quality =('Context Quality (1-5)', 'mean'),
        Avg_Answer_Quality  =('Answer Quality (1-5)',  'mean'),
        Avg_Latency_s       =('Latency (s)',           'mean'),
    )
    .round(3)
    .reset_index()
)

# Sort by dimension size for a clean comparison
size_order = {'small (384d)': 0, 'medium (768d)': 1, 'large (1024d)': 2}
embed_compare['_ord'] = embed_compare['Embedding Model'].map(size_order)
embed_compare = embed_compare.sort_values('_ord').drop(columns='_ord')
print("Embedding model comparison (all strategies, all queries):")
print(embed_compare.to_string(index=False))

Embedding model comparison (all strategies, all queries):
Embedding Model  Avg_Context_Quality  Avg_Answer_Quality  Avg_Latency_s
   small (384d)                2.333               1.467          2.699
  medium (768d)                2.200               1.267          1.843
  large (1024d)                2.867               1.733          2.110


## Embedding Size Analysis

### 1. Retrieval Quality (Avg Context Quality)

| Embedding Model | Avg Context Quality |
|----------------|-------------------|
| Small (384d / multi-qa-MiniLM-L6-cos-v1) | 2.33 |
| Medium (768d / multi-qa-mpnet-base-cos-v1) | 2.20 |
| Large (1024d / BAAI/bge-large-en-v1.5) | 2.87 |

**Small (384d):** Similarity scores were consistently lower and the top-3 chunks often contained off-topic dialogues. The 384-dimensional space is too compressed for nuanced dialogue semantics, so similar-sounding surface words like "help" and "please" pull irrelevant chunks to the top.

**Medium (768d):** Surprisingly, medium performed *worse* than small on context quality (2.20 vs 2.33). MPNet's larger embedding space did not translate to better retrieval on dialogue-format text — for Q2 and Q5 it actually ranked the correct chunk lower than the small model did, suggesting multi-qa-mpnet-base-cos-v1 is less well-calibrated for short conversational snippets than MiniLM.

**Large (1024d):** Best retrieval scores overall. BGE-large is instruction-tuned for retrieval and its 1024-d embeddings separate topic clusters more cleanly, leading to higher cosine scores for truly relevant chunks. The large model's advantage was clearest in fixed_length configs (avg context quality 3.4) where tight chunk boundaries amplified its discriminative power.


### 2. Answer Quality

| Embedding Model | Avg Answer Quality |
|----------------|------------------|
| Small (384d) | 1.47 |
| Medium (768d) | 1.27 |
| Large (1024d) | 1.73 |

Larger embeddings produced better answers overall, but the ceiling was set by **flan-t5-base** as the generation model. The large model's best configuration (fixed_length + large) achieved avg answer quality of **2.6** — the highest of any config — but even with perfect retrieval, flan-t5 frequently repeated context phrases, ignored the highest-ranked chunk, or hallucinated entirely.

Medium was the worst performer on answer quality despite being the middle embedding size, driven by poor recall on Q2 and Q5 where the correct chunk failed to reach rank 1.



### 3. Did Larger Always Win?

**No.** Medium actually underperformed small on both context quality (2.20 vs 2.33) and answer quality (1.27 vs 1.47), making it the weakest embedding model in this experiment overall.

Large won clearly on **fixed_length** chunking (context: 3.4, answer: 2.6) but showed score inflation on **hybrid_strategic** — assigning high cosine scores (0.62–0.76) to large mixed-topic sections without improving answer usefulness. Answer quality remained **1.0** across all hybrid configs regardless of model size.



### 4. Latency

| Embedding Model | Avg Latency (s) |
|----------------|----------------|
| Small (384d) | 2.70 |
| Medium (768d) | 1.84 |
| Large (1024d) | 2.11 |

Latency did **not** scale monotonically with embedding size as expected. Medium was the fastest (1.84s avg), likely because MPNet's architecture is more inference-optimised for this batch size. Large added only modest overhead vs medium (+0.27s avg).

Notably, flan-t5 **generation time** — not encoding — dominated total latency. Small models sometimes produced longer, looping outputs (e.g. the repeated health check-up phrase in Q1) that inflated their generation time beyond what the larger, more precise models produced.


## Step 2.4 — Compare Chunking Strategies

In [23]:
chunk_compare = (
    eval_df
    .groupby('Chunking Strategy')
    .agg(
        Avg_Context_Quality =('Context Quality (1-5)', 'mean'),
        Avg_Answer_Quality  =('Answer Quality (1-5)',  'mean'),
        Avg_Latency_s       =('Latency (s)',           'mean'),
        Num_Chunks          =('Chunking Strategy',     lambda x:
                              {'fixed_length': len(fixed_chunks),
                               'overlapping_paragraph': len(overlap_chunks),
                               'hybrid_strategic': len(hybrid_chunks)}[x.iloc[0]]),
    )
    .round(3)
    .reset_index()
)
print("Chunking strategy comparison (all embedding models, all queries):")
print(chunk_compare.to_string(index=False))

Chunking strategy comparison (all embedding models, all queries):
    Chunking Strategy  Avg_Context_Quality  Avg_Answer_Quality  Avg_Latency_s  Num_Chunks
         fixed_length                2.800               1.933          3.807          33
     hybrid_strategic                1.733               1.000          1.260          11
overlapping_paragraph                2.867               1.533          1.586          31


## Chunking Strategy Analysis

| Chunking Strategy | Avg Context Quality | Avg Answer Quality | Avg Latency (s) | Num Chunks |
|------------------|-------------------|------------------|----------------|------------|
| Fixed Length | 2.80 | 1.93 | 3.81 | 33 |
| Overlapping Paragraph | 2.87 | 1.53 | 1.59 | 31 |
| Hybrid Strategic | 1.73 | 1.00 | 1.26 | 11 |


### 1. Which Strategy Worked Better?

**Fixed-length** chunking produced the best answer quality (1.93), while **overlapping paragraph** edged ahead on context quality (2.87 vs 2.80). **Hybrid strategic** was the weakest strategy by a significant margin — achieving the lowest scores on both metrics and an answer quality of **1.0** across all embedding models.



### 2. Why?

**Fixed-length** chunking was the most consistently reliable strategy. Its shorter chunks (~134 words) create tight semantic units — each chunk typically covers one dialogue or part of one dialogue — making cosine similarity scores more meaningful. When a chunk scores 0.80, it genuinely means that chunk is about the right topic.

**Overlapping paragraph** chunking was competitive, particularly for Q4 (evening plans) where it correctly surfaced the cinema planning dialogue at rank 1 across medium and large configurations. The 50% paragraph overlap helped in cases where a relevant turn spanned a natural paragraph boundary. However, the overlap also created a recurring **contamination problem**: Dialogue 27 (a bus directions dialogue) was consistently retrieved alongside Dialogue 28 (resignation) because they are adjacent paragraphs — the overlap that was meant to preserve context instead created a false semantic neighbour.

**Hybrid/strategic** chunking was the weakest strategy due to two distinct failure modes:

- **Semantic dilution:** A section containing a doctor visit, a keys dialogue, and a vaccination scene produces an embedding that is a blend of all three topics, reducing cosine similarity with any single-topic query.
- **Answer confusion:** Even when the correct section was retrieved at rank 1, flan-t5-base read the *wrong* dialogue within that section — producing answers like *"beautiful"* (from a triplets dialogue in the same section as a health check-up) or *"is the best one?"* (a fragment from a movies section containing the cinema dialogue).



### 3. How Did Chunking Affect Retrieval Relevance?

Fixed-length and overlapping strategies both reliably placed the correct chunk at rank 1 for Q1 (health check-up) across all three embedding models. Hybrid never achieved this — the best it managed was retrieving the *correct section*, which is a weaker guarantee since the relevant dialogue is buried among unrelated content.

For Q3 (computer advice) and Q5 (lost item), hybrid failed to retrieve the correct chunk in **any** configuration. Fixed-length with the large embedding model retrieved the correct chunk at rank 1 for both queries.



### 4. How Did Chunking Affect Final Answers?

The answer quality drop from fixed/overlapping to hybrid was the sharpest difference in the entire experiment:

| Strategy | Best Answer Produced | Avg Answer Quality |
|----------|--------------------|--------------------|
| Fixed Length | *"There's a baseball game on television tonight"* — specific, accurate, grounded | 1.93 |
| Overlapping Paragraph | *"I'd rather not spend a lot of money. What does the paper say about neighborhood theaters?"* — correct and contextual | 1.53 |
| Hybrid Strategic | *"is the best one?"*, *"beautiful"*, *"Excuse me, do you know where the visa office is?"* — hallucinations | 1.00 |

Hybrid answers were almost universally wrong regardless of embedding model, caused by the generation model reading the wrong dialogue within a large mixed-topic section. Fixed-length's shorter, semantically purer chunks gave flan-t5-base a much cleaner signal to generate from.


## Step 2.5 — Data Scaling Experiment

We test our best configuration (overlapping paragraph + large embedding) on:
- **Subset** — 32 paragraphs (current baseline)
- **Extended** — 882 paragraphs (full dataset is 9000 paragraphs and has a really high runtime so we cut it down but still have a meaniniful increase)

**The second code block might take around 6-7 minutes to run**

In [42]:
# slice dataset

from datasets import load_dataset
from typing import List, Dict

df_full = load_dataset("knkarthick/dialogsum", split="train")

def build_paragraphs_from_slice(dataset, start, end):
    paras = []
    for i, row in enumerate(dataset.select(range(start, end))):
        text = row['dialogue'].strip()
        if len(text.split()) >= 80:
            paras.append({
                'text':       text,
                'word_count': len(text.split()),
                'title':      'Dialogue ' + str(start + i + 1),
                'para_idx':   start + i
            })
    return paras

# Baseline — original 32 paragraphs from Part 1
baseline_paras = PARAGRAPHS
large_paras = build_paragraphs_from_slice(df_full, 0, 1100)

print("Baseline : " + str(len(baseline_paras)) + " paragraphs")
print("Extended : " + str(len(large_paras))    + " paragraphs")

Baseline : 32 paragraphs
Extended : 882 paragraphs


In [43]:
# ── Data Scaling Experiment ───────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import time

scale_query = "How does someone handle quitting or leaving a job?"

# Best config: overlapping_paragraph + large
embed_model = SentenceTransformer("BAAI/bge-large-en-v1.5")

def run_scaling_experiment(paras, label, query, embed_model, k=3):
    chunks = []
    for i in range(len(paras) - 1):
        combined = paras[i]['text'] + " " + paras[i+1]['text']
        chunks.append({
            'text':     combined,
            'title':    paras[i]['title'] + " + " + paras[i+1]['title'],
            'chunk_id': i
        })
    if len(paras) == 1:
        chunks = [{'text': paras[0]['text'], 'title': paras[0]['title'], 'chunk_id': 0}]

    # Embed chunks
    chunk_texts      = [c['text'] for c in chunks]
    chunk_embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True)

    # Embed query and retrieve
    t0          = time.time()
    q_emb       = embed_model.encode([query], normalize_embeddings=True)
    scores      = (chunk_embeddings @ q_emb.T).squeeze()
    top_k_idx   = scores.argsort()[::-1][:k]
    latency     = time.time() - t0

    results = []
    for idx in top_k_idx:
        results.append({
            'score': float(scores[idx]),
            'title': chunks[idx]['title'],
            'text':  chunks[idx]['text'][:200]
        })

    # Print results
    print("\n" + "="*70)
    print("SCALE    : " + label)
    print("PARAGRAPHS: " + str(len(paras)) + "  |  CHUNKS: " + str(len(chunks)))
    print("LATENCY  : " + str(round(latency, 3)) + "s")
    print("TOP-3 SCORES: " + str([round(r['score'], 4) for r in results]))
    for i, r in enumerate(results, 1):
        print("\n  Rank " + str(i) + " | score=" + str(round(r['score'], 4)) + " | " + r['title'])
        print("  " + r['text'][:150] + "...")

    return {
        'label':        label,
        'n_paragraphs': len(paras),
        'n_chunks':     len(chunks),
        'avg_top3_sim': float(scores[top_k_idx].mean()),
        'top1_sim':     float(scores[top_k_idx[0]]),
        'latency':      latency,
        'top_chunks':   results
    }

#  Run across all three scales 
scaling_results = []
for paras, label in [
    (small_paras,    "Subset   (~"  + str(len(small_paras))    + " paras)"),
    (baseline_paras, "Baseline (~"  + str(len(baseline_paras)) + " paras)"),
    (large_paras,    "Extended (~"  + str(len(large_paras))    + " paras)"),
]:
    r = run_scaling_experiment(paras, label, scale_query, embed_model)
    scaling_results.append(r)

# Summary t
scale_df = pd.DataFrame([{
    'Scale':         r['label'],
    'Paragraphs':    r['n_paragraphs'],
    'Chunks':        r['n_chunks'],
    'Avg Top-3 Sim': round(r['avg_top3_sim'], 4),
    'Top-1 Sim':     round(r['top1_sim'],     4),
    'Latency (s)':   round(r['latency'],      3),
} for r in scaling_results])

print("\n\nSCALING SUMMARY TABLE")
print(scale_df.to_string(index=False))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SCALE    : Subset   (~10 paras)
PARAGRAPHS: 10  |  CHUNKS: 9
LATENCY  : 0.106s
TOP-3 SCORES: [0.576, 0.5497, 0.4931]

  Rank 1 | score=0.576 | Dialogue 8 + Dialogue 9
  #Person1#: Can I help you?
#Person2#: Yes. I sent in my resume at the end of last week. I'm applying for the accounts assistant position.
#Person1#: M...

  Rank 2 | score=0.5497 | Dialogue 5 + Dialogue 8
  #Person1#: Watsup, ladies! Y'll looking'fine tonight. May I have this dance?
#Person2#: He's cute! He looks like Tiger Woods! But, I can't dance. . .
...

  Rank 3 | score=0.4931 | Dialogue 11 + Dialogue 12
  #Person1#: Could you do me a favor?
#Person2#: Sure. What is it?
#Person1#: Could you run over to the store? We need a few things.
#Person2#: All righ...

SCALE    : Baseline (~32 paras)
PARAGRAPHS: 32  |  CHUNKS: 31
LATENCY  : 0.097s
TOP-3 SCORES: [0.6693, 0.6249, 0.5885]

  Rank 1 | score=0.6693 | Dialogue 28 + Dialogue 29
  #Person1#: Mr. White, I would like to give you notice that I will be leaving the comp

### Query Tested
*"How does someone handle quitting or leaving a job?"*
**Best configuration used:** Overlapping Paragraph + Large (1024d BGE)
- We also ran a 10 paragraph chunk for comparison but it isn't part of the scaling experiment 


### Results Summary

| Scale | Paragraphs | Chunks | Avg Top-3 Sim | Top-1 Sim | Latency (s) |
|-------|-----------|--------|--------------|-----------|-------------|
| Baseline | 32 | 31 | 0.6276 | 0.6693 | 0.097 |
| Extended | 882 | 881 | 0.6775 | 0.6847 | 0.079 |

---

### 1. How Does Dataset Size Affect Retrieval Quality?

Retrieval quality **improved** when scaling from 32 to 882 paragraphs, which is the opposite of what might be expected. Both the avg top-3 similarity (0.6276 → 0.6775) and top-1 similarity (0.6693 → 0.6847) increased with the
larger dataset.

The reason is that the extended dataset contained *more and better* resignationdialogues than the baseline. The baseline only had one resignation dialogue (Dialogue 28), so the system was limited to retrieving that single relevant chunk plus adjacent contamination. The extended set surfaced three genuinely relevant chunks:

- **Rank 1 (0.6847):** Dialogue 1054 + 1055 — explicitly about resigning from
  a current job, scoring *higher* than the baseline's best chunk
- **Rank 2 (0.6740):** Dialogue 924 + 926 — about moving out, tangentially
  related to leaving a situation
- **Rank 3 (0.6739):** Dialogue 752 + 753 — explicitly about handing in a
  resignation letter

This means the large BGE model not only maintained retrieval precision at 28× the data size, it found *better* matches than existed in the baseline.


### 2. How Does Dataset Size Affect Noise?

Noise **decreased** when scaling to the larger dataset, which is again counterintuitive but explainable. In the baseline, rank 2 was Dialogue 27 + 28 : the recurring bus contamination issue where an unrelated dialogue scored highly simply because it was adjacent to the resignation dialogue via the overlap. This contamination disappeared entirely in the extended set,replaced by genuinely relevant resignation and workplace dialogues.

This happened because with 881 chunks to choose from, the BGE model had enough truly relevant options to fill all three top-3 slots. In the baseline, with only 31 chunks, the model was forced to fall back on adjacent-overlap chunks as the next best option.


### Key Takeaways

- **More data improved both retrieval quality and noise** in this experiment
  because the extended dataset contained richer coverage of workplace and
  resignation scenarios.
- **The large BGE model scaled efficiently** — latency actually *decreased*
  slightly (0.097s → 0.079s) from baseline to extended, showing that
  brute-force cosine similarity at this scale is not yet a bottleneck.



